# Worked block analysis — CAN transceiver (Hamilton DAG)

**Now driven by Hamilton.** The same CAN transceiver analysis as before, but the leaf inputs and derived nodes live in proper Python modules (`blocks/can_transceiver/leaves.py` and `analysis.py`) and Hamilton wires them into a DAG by parameter name. The notebook is reduced to *loading the project*, *running the DAG*, and *rendering / spec-checking the results*.

**The block.** A CAN bus transceiver on a 5 V rail. Quantities of interest:

1. **Supply power** — mode-dependent current draw × rail tolerance.
2. **Junction temperature** — `ambient + dissipation × R_θJA`. Ambient is auto-supplied from project scenarios.
3. **Spec checks** — power ≤ 500 mW and T_J ≤ 125 °C across every (scenario × mode) combination, with bounds linked to Jama requirement IDs.

**Pipeline:**

```
scenarios.toml ─┐
modes.toml     ─┤
requirements.py┤───▶ Project.load() ──▶ project.run([leaves, analysis], targets=[...])
blocks/...     ─┘                                          │
                                                           ▼
                                                results: dict[str, Quantity]
                                                           │
                                                           ▼
                                                  rendering / spec checks
```

## 1. Load project + block modules

`Project.load(...)` reads `project/scenarios.toml` and `project/modes.toml` and auto-extracts every context key that appears in *every* scenario (here: `ambient_temp` and `vbat`) into Quantity inputs ready for the DAG.

In [1]:
import sys
from pathlib import Path

# Make blocks/ and project/ importable from the notebook's cwd.
sys.path.insert(0, str(Path.cwd()))

from framework import Project, requirements
from framework.units import V, A, mA, mW, K, degC

project = Project.load(
    scenarios=Path.cwd() / "project" / "scenarios.toml",
    modes=Path.cwd() / "project" / "modes.toml",
)

# Importing project.requirements auto-populates framework.requirements.
from project.requirements import OPERATING_TEMP, CAN_5V_RAIL, VBAT

# Import the block's leaves and analysis modules — Hamilton will discover
# functions in these and build the DAG.
from blocks.can_transceiver import leaves, analysis

print(f"Project: {len(project.scenarios)} scenarios, "
      f"{len(project.modes)} modes, "
      f"{len(requirements.list_all())} requirements")
print(f"Scenarios: {project.scenarios.names()}")
print(f"Modes:     {project.modes.names()}")
print(f"Auto-extracted DAG inputs from project state: "
      f"{[k for k in project.standard_inputs() if k not in ('scenarios', 'modes')]}")

Project: 3 scenarios, 4 modes, 9 requirements
Scenarios: ['nominal', 'cold_low_vin', 'hot_high_vin']
Modes:     ['off', 'sleep', 'active', 'diagnostic']
Auto-extracted DAG inputs from project state: ['ambient_temp', 'vbat']


## 2. The block layout

Per design doc 6.6 a block subpackage looks like:

```
blocks/can_transceiver/
  __init__.py     # (re-exports public Contracts — empty pre-step-7)
  leaves.py       # i_supply, v_supply, r_theta_ja, t_j_max
  analysis.py     # power, thermal_rise, t_j
```

Hamilton builds the DAG from the function signatures: a parameter named `v_supply` wires to the function whose name is `v_supply`. `project.list_nodes(...)` shows every node that ends up in the DAG.

In [2]:
import inspect

print("DAG nodes:", project.list_nodes([leaves, analysis]))
print()

# Inspect each block module to see the function shapes Hamilton sees.
for module, name in ((leaves, "leaves"), (analysis, "analysis")):
    print(f"=== blocks/can_transceiver/{name}.py ===")
    fns = [m for m in dir(module) if callable(getattr(module, m)) and not m.startswith("_")]
    for fn_name in fns:
        fn = getattr(module, fn_name)
        if getattr(fn, "__module__", None) != module.__name__:
            continue  # skip re-exports
        sig = inspect.signature(fn)
        params = ", ".join(sig.parameters)
        print(f"  def {fn_name}({params}) -> Quantity")
    print()

DAG nodes: ['ambient_temp', 'can_5v_rail', 'i_supply', 'power', 'r_theta_ja', 't_j', 't_j_max', 'thermal_rise', 'v_supply']

=== blocks/can_transceiver/leaves.py ===
  def i_supply() -> Quantity
  def r_theta_ja() -> Quantity
  def t_j_max() -> Quantity
  def v_supply(can_5v_rail) -> Quantity

=== blocks/can_transceiver/analysis.py ===
  def power(v_supply, i_supply) -> Quantity
  def t_j(ambient_temp, thermal_rise) -> Quantity
  def thermal_rise(power, r_theta_ja) -> Quantity



## 3. Execute the DAG

`project.run(modules, targets, inputs)` builds the Hamilton driver from the block modules, layers the user-supplied inputs over the project's standard inputs (`ambient_temp`, `vbat`, `scenarios`, `modes`), and computes the requested targets.

`can_5v_rail` is passed explicitly because the `v_supply` leaf needs the `CAN_5V_RAIL` requirement (REQ-PWR-005) — that's how the block ties its supply tolerance to the project's rail spec.

In [3]:
results = project.run(
    modules=[leaves, analysis],
    targets=["power", "thermal_rise", "t_j", "t_j_max"],
    inputs={"can_5v_rail": CAN_5V_RAIL},
)

print("Computed targets:")
for name, q in results.items():
    print(f"  {name:14s} unit = {q.unit}")

Computed targets:
  power          unit = mW
  thermal_rise   unit = K
  t_j            unit = °C
  t_j_max        unit = °C


## 4. Power dissipation result

`power` carries the mode axis (from `i_supply`) with a range per mode (from `v_supply × i_supply` interval arithmetic).

In [4]:
power = results["power"]
mode_names = project.modes.names()
scen_names = project.scenarios.names()

print(f"{'mode':12s} {'min':>10s} {'max':>10s}")
print("-" * 36)
for m in mode_names:
    val = power.at(mode=m)
    lo, hi = val if isinstance(val, tuple) else (val, val)
    print(f"{m:12s} {lo:9.3f}mW {hi:9.3f}mW")

mode                min        max
------------------------------------
off              0.000mW     0.000mW
sleep            0.038mW     0.079mW
active         213.750mW   341.250mW
diagnostic     332.500mW   472.500mW


## 5. Junction temperature result

`t_j` varies along **both axes simultaneously** — `ambient_temp` brought in the scenario axis, `thermal_rise` brought in the mode axis. Every (scenario × mode) cell has its own min/max range.

In [5]:
t_j = results["t_j"]

def cell_str(val):
    if isinstance(val, tuple):
        lo, hi = val
        return f"{lo:5.1f} – {hi:5.1f} °C"
    return f"{val:5.1f} °C"

print(f"{'mode':12s} | " + " | ".join(f"{s:^16s}" for s in scen_names))
print("-" * (14 + 19 * len(scen_names)))
for m in mode_names:
    cells = [cell_str(t_j.at(mode=m, scenario=s)) for s in scen_names]
    print(f"{m:12s} | " + " | ".join(f"{c:^16s}" for c in cells))

mode         |     nominal      |   cold_low_vin   |   hot_high_vin  
-----------------------------------------------------------------------
off          |      25.0 °C     |     -40.0 °C     |      85.0 °C    
sleep        |  25.0 –  25.0 °C | -40.0 – -40.0 °C |  85.0 –  85.0 °C
active       |  50.6 –  65.9 °C | -14.4 –   0.9 °C | 110.6 – 125.9 °C
diagnostic   |  64.9 –  81.7 °C |  -0.1 –  16.7 °C | 124.9 – 141.7 °C


## 6. Spec checks

`.within(lo, hi)` returns `True` iff **every** scenario/mode evaluation lies inside the spec. If you only need a yes/no for CI, this is your friend; for narrow margins or design-review reports we'll dig into the worst cell explicitly.

In [6]:
power_ok = power.within(0 * mW, 500 * mW)

# Lower T_J bound tied to project requirement REQ-ENV-001 (OPERATING_TEMP);
# upper bound is the block-local datasheet derate (t_j_max from leaves).
t_j_max = results["t_j_max"]
t_j_ok = t_j.within(OPERATING_TEMP.min, t_j_max)

print(f"Power dissipation <= 500 mW everywhere?           {power_ok}")
print(f"Junction temp within [{OPERATING_TEMP.min:~P}, {t_j_max.at():.0f} C]?   {t_j_ok}")
print(f"  (linked to {OPERATING_TEMP.req} / block-local datasheet derate)")

# When something fails, surface the worst-case corner so the engineer knows
# where to look. Step 9 (VerificationTest) will do this automatically with
# margin-to-spec and a Jama link; here we just iterate.
if not t_j_ok:
    print("\nT_J failures:")
    for m in mode_names:
        for s in scen_names:
            val = t_j.at(mode=m, scenario=s)
            hi = val[1] if isinstance(val, tuple) else val
            if hi > 125.0:
                print(f"  mode={m:11s} scenario={s:14s} T_J max = {hi:5.1f} C   (over by {hi - 125:.1f} C)")

Power dissipation <= 500 mW everywhere?           True
Junction temp within [-40.0 °C, 125 C]?   False
  (linked to REQ-ENV-001 / block-local datasheet derate)

T_J failures:
  mode=active      scenario=hot_high_vin   T_J max = 125.9 C   (over by 0.9 C)
  mode=diagnostic  scenario=hot_high_vin   T_J max = 141.7 C   (over by 16.7 C)


## 7. What changed (and what didn't) vs. the inline version

This notebook used to hand-compute power and junction temp inline. Now it imports two modules — `blocks/can_transceiver/leaves.py` and `blocks/can_transceiver/analysis.py` — and asks Hamilton to wire them together into a DAG.

What changed:

- **The leaf inputs and derived nodes are reusable Python modules**, not throwaway cells. Verification tests (step 9) and the eventual `report.ipynb` template can both consume the same functions.
- **The DAG is introspectable** — `project.list_nodes(...)` enumerates everything Hamilton sees; future tooling will let you visualize it and walk provenance.
- **Auto-supplied inputs** — `ambient_temp` is extracted from scenarios automatically, no per-block boilerplate.
- **Inputs are layered** — project-derived inputs first, user explicit overrides on top.

What did *not* change:

- The Quantity arithmetic. `(v_supply * i_supply).to(mW)` looks the same whether it runs in a notebook cell or a Hamilton node.
- The spec-check semantics. `.within(...)` is still how we ask "is this OK?".
- The thermal failure. T_J still maxes out at 141.7 °C in `hot_high_vin / diagnostic` — surfacing real worst-case behavior, now traceable from `REQ-ENV-001` to a function in `analysis.py`.

**Still to come (Section 14):**

- Step 4b: content-addressed caching — DAG nodes whose inputs haven't changed return cached Quantities.
- Step 5: typed component library — `MOSFET`, `Resistor`, `Capacitor`, etc. as Pydantic schemas pulled into leaves.
- Step 7: `Contract` type and cross-block wiring — `v_supply` becomes a Contract imported from `blocks/power_supply` instead of pulled from a Requirement.
- Step 9: `@verification_test` replaces the manual `within(...)` block — same logic, but reportable to Jama and the PR comment bot.